In [1]:
import sys
print(sys.executable)

d:\Code\swe_agent_jom\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path)

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", deepseek_api_key is not None)

DEEPSEEK_API_KEY loaded: True


In [3]:
from __future__ import annotations

import json
import os
from collections.abc import Mapping
from datetime import date
from typing import cast

from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import (
    ChatCompletionAssistantMessageParam,
    ChatCompletionMessageParam,
    ChatCompletionSystemMessageParam,
    ChatCompletionToolMessageParam,
    ChatCompletionToolParam,
    ChatCompletionUserMessageParam,
)
from openai.types.chat.chat_completion_message_function_tool_call import (
    ChatCompletionMessageFunctionToolCall,
)


# =========================
# 1. 基础配置
# =========================

load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

if DEEPSEEK_API_KEY is None:
    raise RuntimeError("Missing environment variable: DEEPSEEK_API_KEY")

MODEL = "deepseek-v4-flash"

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com",
)


# =========================
# 2. 业务状态：生日表
# =========================

birthday_table: dict[str, str] = {
    "Jack": "3月18日",
    "Alice": "7月2日",
    "Bob": "11月9日",
    "Tom": "1月25日",
}


MAX_DAY_BY_MONTH: dict[int, int] = {
    1: 31,
    2: 29,
    3: 31,
    4: 30,
    5: 31,
    6: 30,
    7: 31,
    8: 31,
    9: 30,
    10: 31,
    11: 30,
    12: 31,
}


# =========================
# 3. 工具函数的参数校验
# =========================

def validate_month_day(month: int, day: int) -> None:
    if month not in MAX_DAY_BY_MONTH:
        raise ValueError(f"Invalid month: {month}")

    max_day = MAX_DAY_BY_MONTH[month]

    if day < 1 or day > max_day:
        raise ValueError(f"Invalid day: {month}月 does not have day {day}")


def format_birthday(month: int, day: int) -> str:
    validate_month_day(month, day)
    return f"{month}月{day}日"


def parse_json_object(raw_json: str) -> dict[str, object]:
    try:
        data: object = json.loads(raw_json)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Tool arguments are not valid JSON: {raw_json}") from exc

    if not isinstance(data, dict):
        raise ValueError("Tool arguments must be a JSON object.")

    return cast(dict[str, object], data)


def get_required_str(arguments: Mapping[str, object], key: str) -> str:
    value = arguments.get(key)

    if not isinstance(value, str):
        raise ValueError(f"Argument '{key}' must be a string.")

    value = value.strip()

    if value == "":
        raise ValueError(f"Argument '{key}' cannot be empty.")

    return value


def get_required_int(arguments: Mapping[str, object], key: str) -> int:
    value = arguments.get(key)

    if isinstance(value, bool):
        raise ValueError(f"Argument '{key}' must be an integer, not boolean.")

    if isinstance(value, int):
        return value

    if isinstance(value, float) and value.is_integer():
        return int(value)

    if isinstance(value, str):
        try:
            return int(value)
        except ValueError as exc:
            raise ValueError(f"Argument '{key}' must be an integer.") from exc

    raise ValueError(f"Argument '{key}' must be an integer.")


# =========================
# 4. 三个真实工具函数
# =========================

def read_birthday(name: str) -> dict[str, object]:
    """
    读取某个人的生日。
    """

    birthday = birthday_table.get(name)

    if birthday is None:
        return {
            "ok": False,
            "message": f"没有找到 {name} 的生日。",
            "name": name,
        }

    return {
        "ok": True,
        "name": name,
        "birthday": birthday,
    }


def find_people_by_birthday(month: int, day: int) -> dict[str, object]:
    """
    根据日期查当天生日的人。
    """

    target_birthday = format_birthday(month, day)

    people = [
        name
        for name, birthday in birthday_table.items()
        if birthday == target_birthday
    ]

    return {
        "ok": True,
        "birthday": target_birthday,
        "people": people,
        "count": len(people),
    }


def update_birthday(name: str, month: int, day: int) -> dict[str, object]:
    """
    修改或新增某个人的生日。
    """

    new_birthday = format_birthday(month, day)
    old_birthday = birthday_table.get(name)

    birthday_table[name] = new_birthday

    return {
        "ok": True,
        "name": name,
        "old_birthday": old_birthday,
        "new_birthday": new_birthday,
        "birthday_table": birthday_table.copy(),
    }


# =========================
# 5. 三个 Tool Schema
# =========================

TOOLS: list[ChatCompletionToolParam] = [
    {
        "type": "function",
        "function": {
            "name": "read_birthday",
            "description": (
                "Read one person's birthday from the birthday table. "
                "Use this when the user asks for a specific person's birthday."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "The person's exact name, for example Jack, Alice, Bob, or Tom.",
                    },
                },
                "required": ["name"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "find_people_by_birthday",
            "description": (
                "Find all people whose birthday is on a specific month and day. "
                "Use this when the user asks who has a birthday on a given date."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "month": {
                        "type": "integer",
                        "description": "Month number from 1 to 12.",
                        "minimum": 1,
                        "maximum": 12,
                    },
                    "day": {
                        "type": "integer",
                        "description": "Day number in the month.",
                        "minimum": 1,
                        "maximum": 31,
                    },
                },
                "required": ["month", "day"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "update_birthday",
            "description": (
                "Update or create one person's birthday in the birthday table. "
                "Use this when the user asks to modify, change, update, or add a birthday."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "The person's exact name.",
                    },
                    "month": {
                        "type": "integer",
                        "description": "New birthday month number from 1 to 12.",
                        "minimum": 1,
                        "maximum": 12,
                    },
                    "day": {
                        "type": "integer",
                        "description": "New birthday day number.",
                        "minimum": 1,
                        "maximum": 31,
                    },
                },
                "required": ["name", "month", "day"],
                "additionalProperties": False,
            },
        },
    },
]


# =========================
# 6. Tool Dispatcher
# =========================

def dispatch_tool(tool_name: str, arguments: Mapping[str, object]) -> dict[str, object]:
    """
    根据模型返回的 tool_name 和 arguments，执行真实 Python 函数。

    这里不要盲目信任模型生成的参数。
    即使 schema 已经限制了参数格式，后端仍然需要自己校验。
    """

    try:
        if tool_name == "read_birthday":
            name = get_required_str(arguments, "name")
            return read_birthday(name=name)

        if tool_name == "find_people_by_birthday":
            month = get_required_int(arguments, "month")
            day = get_required_int(arguments, "day")
            return find_people_by_birthday(month=month, day=day)

        if tool_name == "update_birthday":
            name = get_required_str(arguments, "name")
            month = get_required_int(arguments, "month")
            day = get_required_int(arguments, "day")
            return update_birthday(name=name, month=month, day=day)

        return {
            "ok": False,
            "error": f"Unknown tool: {tool_name}",
        }

    except Exception as exc:
        return {
            "ok": False,
            "error": str(exc),
            "tool_name": tool_name,
            "arguments": dict(arguments),
        }


# =========================
# 7. 类型安全地构造 assistant tool-call message
# =========================

def build_assistant_message_param(
    content: str | None,
    tool_calls: list[ChatCompletionMessageFunctionToolCall],
) -> ChatCompletionAssistantMessageParam:
    """
    OpenAI SDK 返回的 message 是响应对象，不适合直接 append 到 messages 里。

    为了减少类型检查报错，这里手动构造 ChatCompletionAssistantMessageParam。
    """

    assistant_message = {
        "role": "assistant",
        "content": content,
        "tool_calls": [
            {
                "id": tool_call.id,
                "type": "function",
                "function": {
                    "name": tool_call.function.name,
                    "arguments": tool_call.function.arguments,
                },
            }
            for tool_call in tool_calls
        ],
    }

    return cast(ChatCompletionAssistantMessageParam, assistant_message)


# =========================
# 8. Agent 主循环
# =========================

class BirthdayAgent:
    def __init__(self) -> None:
        today = date.today()

        system_prompt = (
            "你是一个生日表助手。"
            "你不能凭空回答生日数据。"
            "当用户询问、查询、修改生日时，你必须优先调用工具。"
            f"今天的日期是 {today.month}月{today.day}日。"
            "如果用户说“今天生日的人”，你可以根据今天的日期调用 find_people_by_birthday。"
            "回答用户时要简洁，并用中文回答。"
        )

        system_message: ChatCompletionSystemMessageParam = {
            "role": "system",
            "content": system_prompt,
        }

        self.messages: list[ChatCompletionMessageParam] = [system_message]

    def ask(self, user_input: str) -> str:
        user_message: ChatCompletionUserMessageParam = {
            "role": "user",
            "content": user_input,
        }

        self.messages.append(user_message)

        max_tool_rounds = 5

        for _ in range(max_tool_rounds):
            completion = client.chat.completions.create(
                model=MODEL,
                messages=self.messages,
                tools=TOOLS,
                tool_choice="auto",
            )

            message = completion.choices[0].message

            if message.tool_calls is None:
                return message.content or ""

            function_tool_calls: list[ChatCompletionMessageFunctionToolCall] = []

            for tool_call in message.tool_calls:
                if tool_call.type != "function":
                    raise RuntimeError(f"Unsupported tool call type: {tool_call.type}")

                function_tool_calls.append(
                    cast(ChatCompletionMessageFunctionToolCall, tool_call)
                )

            assistant_message = build_assistant_message_param(
                content=message.content,
                tool_calls=function_tool_calls,
            )

            self.messages.append(assistant_message)

            for tool_call in function_tool_calls:
                tool_name = tool_call.function.name
                raw_arguments = tool_call.function.arguments
                arguments = parse_json_object(raw_arguments)

                tool_result = dispatch_tool(
                    tool_name=tool_name,
                    arguments=arguments,
                )

                tool_message: ChatCompletionToolMessageParam = {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result, ensure_ascii=False),
                }

                self.messages.append(tool_message)

        return "工具调用轮次过多，已停止。"


# =========================
# 9. Demo
# =========================

def main() -> None:
    agent = BirthdayAgent()

    demo_questions = [
        "Jack 的生日是什么？",
        "7月2日当天有哪些人过生日？",
        "把 Bob 的生日改成 12月10日。",
        "Bob 现在的生日是什么？",
        "今天有哪些人过生日？",
    ]

    for question in demo_questions:
        print(f"\nUser: {question}")
        answer = agent.ask(question)
        print(f"Assistant: {answer}")

    print("\n当前 Python 内存里的 birthday_table：")
    print(birthday_table)


if __name__ == "__main__":
    main()


User: Jack 的生日是什么？
Assistant: Jack 的生日是 **3月18日** 🎂

User: 7月2日当天有哪些人过生日？
Assistant: 7月2日过生日的人是 **Alice**。🎂

User: 把 Bob 的生日改成 12月10日。
Assistant: 已经将 Bob 的生日从 11月9日 改为 **12月10日**。

User: Bob 现在的生日是什么？
Assistant: Bob 现在的生日是 **12月10日**。

User: 今天有哪些人过生日？
Assistant: 今天（7月7日）没有人生日。

当前 Python 内存里的 birthday_table：
{'Jack': '3月18日', 'Alice': '7月2日', 'Bob': '12月10日', 'Tom': '1月25日'}
